# HAMIC — TPU readiness drill

Run the two phases in order, restarting the kernel between them:

1. `PHASE = "train"`: fetch the smoke corpus, train 6 steps, publish the step-4 checkpoint.
2. `PHASE = "resume"`: restore step 4 from the Hub, finish step 12, compare the resumed losses
   with an uninterrupted in-memory reference.

A completed run cannot be resumed; use a new `RUN_ID` for a fresh drill. The token is read as
`HF_KEY` (TPU only). Do not type it into a saved cell.

In [ ]:
# Any Git ref works: a branch (main), a tag, or a commit SHA.
HAMIC_REF = "main"
HUB_REPO = "HaianCao/hamic-runs"
DATA_REPO = "HaianCao/hamic-data"
RUN_ID = "tpu-drill-001"
XET_HIGH_PERFORMANCE = True

# "train" for session 1, "resume" after a kernel restart.
PHASE = "train"
STEPS = 12
STOP_AFTER = 6
CHECKPOINT_EVERY = 4

In [ ]:
import json
import os
import shutil
import sys
import tarfile
import tempfile
import urllib.request
from pathlib import Path

os.environ["HF_XET_HIGH_PERFORMANCE"] = "1" if XET_HIGH_PERFORMANCE else "0"
work_root = Path(tempfile.gettempdir()) / "hamic-tpu"
source_dir = work_root / "source"
stamp = work_root / "source-ref.json"
repo = "https://github.com/HaianCao/HAMIC-LM"
with urllib.request.urlopen(
    f"https://api.github.com/repos/HaianCao/HAMIC-LM/commits/{HAMIC_REF}",
    timeout=30,
) as response:
    commit = json.load(response)["sha"]
stamp_data = json.loads(stamp.read_text()) if stamp.is_file() else None
if not isinstance(stamp_data, dict):
    stamp_data = None
if (
    not source_dir.is_dir()
    or stamp_data is None
    or stamp_data.get("ref") != HAMIC_REF
    or stamp_data.get("commit") != commit
):
    archive = work_root / "source.tar.gz"
    work_root.mkdir(parents=True, exist_ok=True)
    urllib.request.urlretrieve(f"{repo}/archive/{commit}.tar.gz", archive)
    unpack = work_root / "unpack"
    shutil.rmtree(unpack, ignore_errors=True)
    unpack.mkdir()
    with tarfile.open(archive) as tar:
        try:
            tar.extractall(unpack, filter="data")
        except TypeError:
            tar.extractall(unpack)
    (top,) = unpack.iterdir()
    shutil.rmtree(source_dir, ignore_errors=True)
    shutil.move(str(top), source_dir)
    shutil.rmtree(unpack, ignore_errors=True)
    archive.unlink()
    stamp.write_text(json.dumps({"ref": HAMIC_REF, "commit": commit}))
sys.path.insert(0, str(source_dir / "src"))
if "hamic_jax" in sys.modules:
    raise RuntimeError(
        "An older hamic_jax is already imported. Restart the kernel and run this notebook again."
    )
import hamic_jax

if Path(hamic_jax.__file__).resolve().parent != (source_dir / "src" / "hamic_jax").resolve():
    raise RuntimeError("hamic_jax was imported from outside the pinned source tree")
print(f"Source {HAMIC_REF} resolved to {commit[:12]} · work root {work_root}")
print("The wheel excludes src/hamic_jax, so this notebook imports the pinned checkout.")

In [ ]:
from hamic_jax.dependencies import ensure_dependencies

installed = ensure_dependencies()
print("Runtime dependencies are available." if not installed else f"Installed: {', '.join(installed)}")

In [ ]:
import jax

backend = jax.default_backend()
if backend != "tpu":
    raise RuntimeError(
        f"No TPU attached (jax.default_backend()={backend!r}). Attach the TPU and rerun."
    )
devices = jax.devices()
print(
    f"JAX {jax.__version__} · backend {backend} · {len(devices)} device(s) "
    f"{devices[0].device_kind} · code {commit[:12]}"
)

In [ ]:
import getpass

hub_token = os.environ.get("HF_KEY", "").strip()
if not hub_token:
    hub_token = getpass.getpass("Hugging Face token (HF_KEY): ").strip()
if not hub_token:
    raise RuntimeError("Enter a Hugging Face token in HF_KEY or at the prompt.")

In [ ]:
from hamic_jax.checkpoint import HuggingFaceObjectStore

run_store = HuggingFaceObjectStore(HUB_REPO, private=True, token=hub_token)
data_store = HuggingFaceObjectStore(DATA_REPO, repo_type="dataset", create=False, token=hub_token)
print(f"Hub account {run_store.whoami()} · runs {HUB_REPO} · data {DATA_REPO}")

In [ ]:
from hamic_jax.drill import build_drill_config
from hamic_jax.presets import load_training_config

base_config, preset = load_training_config(
    source_dir / "configs" / "train" / "smoke-cpu-t1.yaml"
)
drill_config = build_drill_config(
    base_config,
    run_id=RUN_ID,
    steps=STEPS,
    checkpoint_every=CHECKPOINT_EVERY,
)
print(f"Drill {drill_config.run_id}: {STEPS} steps, stop after {STOP_AFTER}")

In [ ]:
from hamic_jax.lab import fetch_corpus

drill_config, corpus = fetch_corpus(drill_config, data_store, work_root, drill_config.run_id)
split_tokens = {name: value["tokens"] for name, value in corpus.manifest["splits"].items()}
print(f"Verified {corpus.root} · train tokens {split_tokens['train']:,}")

In [ ]:
from IPython.display import display
from pathlib import Path

from tqdm.auto import tqdm

from hamic_jax.drill import compare_drill, run_drill_phase
from hamic_jax.plots import drill_comparison, loss_curve

with tqdm(total=STEPS, desc=f"Drill {PHASE}", unit="step") as bar:
    def progress(step, target, loss):
        bar.update(max(step - bar.n, 0))
        bar.set_postfix_str(f"loss {loss:.5f}")

    result = run_drill_phase(
        drill_config,
        preset,
        corpus,
        run_store,
        work_root / "drill" / drill_config.run_id,
        phase=PHASE,
        steps=STEPS,
        stop_after=STOP_AFTER,
        checkpoint_every=CHECKPOINT_EVERY,
        source_commit=commit,
        progress=progress,
        log=print,
    )

print(
    f"Phase {PHASE} · step {result.step}/{STEPS} · resumed from {result.resumed_from_step} "
    f"· completed {result.completed} · checkpoints {[Path(p).name for p in result.checkpoints]}"
)

if PHASE == "resume":
    comparison = compare_drill(
        drill_config,
        preset,
        corpus,
        steps=STEPS,
        stop_after=STOP_AFTER,
        checkpoint_every=CHECKPOINT_EVERY,
        resumed_losses=result.losses,
        resumed_from_step=result.resumed_from_step,
    )
    rows = [
        {
            "step": comparison.start + index + 1,
            "resumed": loss,
            "reference": comparison.reference[comparison.start + index],
            "equal": abs(loss - comparison.reference[comparison.start + index]) <= 1e-6,
        }
        for index, loss in enumerate(comparison.observed)
    ]
    print("Drill passed" if comparison.passed else "Drill FAILED")
    display(rows)
    display(drill_comparison(comparison))
else:
    print(f"Published checkpoints: {result.checkpoints}")
    print("Restart the kernel, set PHASE = 'resume', and run the cells again.")
    display(loss_curve(result, title=f"Drill {PHASE} · {result.run_id}"))